# 훈련량 예측

5장에서 저장한 평가 데이터를 바탕으로, 추천일 이전 기록을 이용해
추천일 당일부터 이후 6일까지의 7일 TSS 합계를 예측한다.

이 예측은 앞으로 기록될 훈련량을 대상으로 하며, 실제 피로·회복
상태나 개별 훈련 처방을 확정하지 않는다.


In [2]:
from pathlib import Path

import pandas as pd

project_root = Path("..").resolve()

evaluation_path = (
    project_root
    / "data"
    / "processed"
    / "goldencheetah_recommendation_evaluation.csv"
)

evaluation_df = pd.read_csv(
    evaluation_path,
    parse_dates=["recommendation_date"],
)

evaluation_df.shape

(100, 18)

In [3]:
evaluation_df = (
    evaluation_df
    .sort_values("recommendation_date")
    .reset_index(drop=True)
)

evaluation_df["recommendation_date"].agg(
    ["min", "max"]
)

min   2009-03-21 00:00:00+00:00
max   2009-06-28 00:00:00+00:00
Name: recommendation_date, dtype: datetime64[us, UTC]

## 1. 입력 후보와 정답값 선택

입력 후보는 추천일 이전 기록으로 계산한 이전 7일·28일 TSS,
이전 7일 운동 시간·라이드 횟수, 전날까지의 연속 라이드 기록일과
연속 무기록일 수로 선택한다. 무기록일은 실제 휴식일로 단정하지 않는다.

정답값 `next_7_day_tss`는 추천일 당일부터 7일간 기록된 TSS 합계다.
이후 TSS 변화량·비율은 미래 기록을 포함하므로 입력에서 제외한다.
부하 라벨과 안내 문장도 이번 입력 후보에서는 제외한다.


In [4]:
feature_columns = [
    "previous_7_day_tss",
    "previous_28_day_tss",
    "previous_7_day_workout_hours",
    "previous_7_day_ride_count",
    "previous_day_recorded_ride_streak_days",
    "previous_day_no_record_streak_days",
]

target_column = "next_7_day_tss"

prediction_df = evaluation_df[
    ["recommendation_date"]
    + feature_columns
    + [target_column]
].copy()

prediction_df.head().round(
    {
        "previous_7_day_tss": 2,
        "previous_28_day_tss": 2,
        "previous_7_day_workout_hours": 2,
        "next_7_day_tss": 2,
    }
)

,recommendation_date,previous_7_day_tss,previous_28_day_tss,previous_7_day_workout_hours,previous_7_day_ride_count,previous_day_recorded_ride_streak_days,previous_day_no_record_streak_days,next_7_day_tss
0,2009-03-21 00:00:00+00:00,829.25,2506.27,11.19,8.0,1.0,0.0,435.28
1,2009-03-22 00:00:00+00:00,926.31,2663.95,12.62,8.0,2.0,0.0,391.33
2,2009-03-23 00:00:00+00:00,812.04,2417.65,10.73,6.0,3.0,0.0,458.54
3,2009-03-24 00:00:00+00:00,709.89,2417.65,9.16,5.0,0.0,1.0,630.81
4,2009-03-25 00:00:00+00:00,580.91,2417.65,7.71,4.0,0.0,2.0,630.81


In [5]:
prediction_df[
    feature_columns + [target_column]
].info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   previous_7_day_tss                      100 non-null    float64
 1   previous_28_day_tss                     100 non-null    float64
 2   previous_7_day_workout_hours            100 non-null    float64
 3   previous_7_day_ride_count               100 non-null    float64
 4   previous_day_recorded_ride_streak_days  100 non-null    float64
 5   previous_day_no_record_streak_days      100 non-null    float64
 6   next_7_day_tss                          100 non-null    float64
dtypes: float64(7)
memory usage: 5.6 KB


In [7]:
prediction_df[
    feature_columns + [target_column]
].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
previous_7_day_tss,100.0,705.09,143.53,362.95,606.98,716.42,821.39,1012.40
previous_28_day_tss,100.0,2800.49,270.56,2213.47,2636.21,2842.41,3017.11,3309.16
previous_7_day_workout_hours,100.0,9.88,2.24,5.16,7.97,9.78,11.58,14.86
previous_7_day_ride_count,100.0,5.43,1.39,3.00,4.00,5.00,6.00,8.00
previous_day_recorded_ride_streak_days,100.0,2.04,2.49,0.00,0.00,1.00,3.00,12.00
previous_day_no_record_streak_days,100.0,0.51,0.86,0.00,0.00,0.00,1.00,4.00
next_7_day_tss,100.0,687.13,155.72,362.95,581.91,706.32,802.62,1012.40


In [8]:
prediction_df[
    "recommendation_date"
].diff().value_counts()

recommendation_date
1 days    99
Name: count, dtype: int64

In [9]:
evaluation_start_date = prediction_df[
    "recommendation_date"
].iloc[80]

training_cutoff_date = (
    evaluation_start_date
    - pd.Timedelta(days=6)
)

train_df = prediction_df.loc[
    prediction_df["recommendation_date"]
    < training_cutoff_date
].copy()

test_df = prediction_df.loc[
    prediction_df["recommendation_date"]
    >= evaluation_start_date
].copy()

train_df.shape, test_df.shape

((74, 8), (20, 8))

In [10]:
baseline_predictions = test_df[
    "previous_7_day_tss"
]

baseline_absolute_errors = (
    test_df[target_column]
    - baseline_predictions
).abs()

baseline_mae = baseline_absolute_errors.mean()

baseline_mae

np.float64(164.30677149999997)

### 예측 데이터 점검 및 기준선 평가 결과

- 5장에서 저장한 100행 18열의 평가 데이터를 불러와, 추천일과 입력 후보 여섯 개, 정답값으로 구성된 100행 8열의 예측용 표를 만들었다.
- 입력 후보는 이전 7일·28일 TSS, 이전 7일 운동 시간·라이드 횟수, 전날까지의 연속 라이드 기록일·연속 무기록일 수로 선택했다. 정답값은 추천일 당일부터 이후 6일까지의 TSS 합계인 `next_7_day_tss`다.
- 입력 후보와 정답값은 모두 숫자형 `float64`였으며, 각 열의 100개 값에 결측값이 없었다. 요약 통계에서 음수나 제거 근거가 있는 명백한 비정상 값은 발견하지 않았다. 연속 라이드 기록일의 최댓값 12일은 기존 분석에서 확인한 기록이므로 유지했다.
- 추천일은 2009년 3월 21일부터 6월 28일까지 하루 간격으로 이어졌다. 앞쪽 74일을 학습 기간으로, 마지막 20일을 평가 기간으로 나누었다. 학습용 7일 정답 기간이 평가 시작일에 걸치지 않도록 평가 직전 6일은 학습에서 제외했다.
- 이후 7일 TSS가 이전 7일 TSS와 같다고 예측하는 단순 예측 기준선(baseline)을 만들었다. 평가 기간 20일의 평균 절대오차(MAE)는 약 164.31 TSS였다.
- 선형회귀 모델은 아직 학습하지 않았다. 다음 단계에서 같은 평가 기간의 MAE를 계산해 기준선과 비교한다.

이 기준선은 앞으로 기록될 훈련량의 예측 성능을 비교하기 위한 것이다. 실제 피로·회복 상태나 개별 훈련 처방의 적절성을 평가하지 않으며, 무기록일을 실제 휴식일로 단정하지 않는다.
